# ⚙️ Setup

In [13]:
#!pip install nltk


In [ ]:
import pandas as pd
import numpy as np
import string

# 📊 Visualization 
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

# 📜 Text preprocessing
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


True

# 🧹 Clean Data


## 📥 Step 1: Load the Dataset

We will load the dataset and preview its structure.


In [15]:

# Load the new uploaded amazon dataset
df = pd.read_csv('datasets/amazon.csv')

# Display the first few rows and column names again to ensure correct structure
df.head(), df.columns
df.head().iloc[0]


product_id                                                    B07JW9H4J1
product_name           Wayona Nylon Braided USB to Lightning Fast Cha...
category               Computers&Accessories|Accessories&Peripherals|...
discounted_price                                                    ₹399
actual_price                                                      ₹1,099
discount_percentage                                                  64%
rating                                                               4.2
rating_count                                                      24,269
about_product          High Compatibility : Compatible With iPhone 12...
user_id                AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...
user_name              Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...
review_id              R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...
review_title           Satisfied,Charging is really fast,Value for mo...
review_content         Looks durable Charging is fi

## ✂️ Step 2: Filter Relevant Columns

We are only interested in:
- `product_name`
- `actual_price`
- `about_product`


In [16]:
# Select the relevant columns
products = df[['product_name', 'actual_price', 'about_product']].copy()


## 💲 Step 3: Clean `actual_price`

Convert `actual_price` from string (e.g., "₹1,099") to integer (e.g., 1099).


In [17]:
products['actual_price'] = (
    products['actual_price']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.extract('(\d+)', expand=False) 
    .astype(float)
    .astype('Int64') 
)
products['actual_price'] = round(products['actual_price'] * 0.012, 2)
products.head(5)

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
C:\Users\hanse.HPUTH99\AppData\Local\Temp\ipykernel_23852\2953457149.py:6: SyntaxWarning: invalid escape sequence '\d'
  .str.extract('(\d+)', expand=False)


,product_name,actual_price,about_product
0,Wayona Nylon Braided USB to Lightning Fast Cha...,13.19,High Compatibility : Compatible With iPhone 12...
1,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,4.19,"Compatible with all Type C enabled devices, be..."
2,Sounce Fast Phone Charging Cable & Data Sync U...,22.79,【 Fast Charger& Data Sync】-With built-in safet...
3,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,8.39,The boAt Deuce USB 300 2 in 1 cable is compati...
4,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,4.79,[CHARGE & SYNC FUNCTION]- This cable comes wit...


## ❌ Step 4: Drop Missing Data

Remove any rows with missing values in the selected columns.


In [18]:
products_cleaned = products.dropna(subset=['product_name', 'actual_price', 'about_product'])
products.isnull().sum()

product_name     0
actual_price     0
about_product    0
dtype: int64

## 🔤 Step 5: Text Preprocessing

We will lowercase and remove punctuation from both `product_name` and `about_product`.


In [19]:
def clean_text(text):
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.lower()

products_cleaned['product_name'] = products_cleaned['product_name'].apply(clean_text)
products_cleaned['about_product'] = products_cleaned['about_product'].apply(clean_text)

stop_words = set(stopwords.words('english'))

In [20]:
def word_tokenizer(text):
    return text.split(' ')

In [21]:
#tokenize, remove stop words, and stem summaries
ps = PorterStemmer()
products_cleaned['tokenized_about'] = products_cleaned['about_product'].apply(word_tokenizer)
products_cleaned['remove_stop_about'] = products_cleaned['tokenized_about'].apply(lambda x: [item for item in x if item not in stop_words])
products_cleaned['remove_stop_stem_about'] = products_cleaned['remove_stop_about'].apply(lambda x: [ps.stem(y) for y in x])

In [22]:
products_cleaned['tokenized_name'] = products_cleaned['product_name'].apply(word_tokenizer)
products_cleaned['remove_stop_name'] = products_cleaned['tokenized_name'].apply(lambda x: [item for item in x if item not in stop_words])
products_cleaned['remove_stop_stem_name'] = products_cleaned['remove_stop_name'].apply(lambda x: [ps.stem(y) for y in x])

In [23]:
products_cleaned

,product_name,actual_price,about_product,tokenized_about,remove_stop_about,remove_stop_stem_about,tokenized_name,remove_stop_name,remove_stop_stem_name
0,wayona nylon braided usb to lightning fast cha...,13.19,high compatibility compatible with iphone 12 ...,"[high, compatibility, , compatible, with, ipho...","[high, compatibility, , compatible, iphone, 12...","[high, compat, , compat, iphon, 12, 11, xxsmax...","[wayona, nylon, braided, usb, to, lightning, f...","[wayona, nylon, braided, usb, lightning, fast,...","[wayona, nylon, braid, usb, lightn, fast, char..."
1,ambrane unbreakable 60w 3a fast charging 15m ...,4.19,compatible with all type c enabled devices be ...,"[compatible, with, all, type, c, enabled, devi...","[compatible, type, c, enabled, devices, androi...","[compat, type, c, enabl, devic, android, smart...","[ambrane, unbreakable, 60w, , 3a, fast, chargi...","[ambrane, unbreakable, 60w, , 3a, fast, chargi...","[ambran, unbreak, 60w, , 3a, fast, charg, 15m,..."
2,sounce fast phone charging cable data sync us...,22.79,【 fast charger data sync】with builtin safety p...,"[【, fast, charger, data, sync】with, builtin, s...","[【, fast, charger, data, sync】with, builtin, s...","[【, fast, charger, data, sync】with, builtin, s...","[sounce, fast, phone, charging, cable, , data,...","[sounce, fast, phone, charging, cable, , data,...","[sounc, fast, phone, charg, cabl, , data, sync..."
3,boat deuce usb 300 2 in 1 typec micro usb str...,8.39,the boat deuce usb 300 2 in 1 cable is compati...,"[the, boat, deuce, usb, 300, 2, in, 1, cable, ...","[boat, deuce, usb, 300, 2, 1, cable, compatibl...","[boat, deuc, usb, 300, 2, 1, cabl, compat, sma...","[boat, deuce, usb, 300, 2, in, 1, typec, , mic...","[boat, deuce, usb, 300, 2, 1, typec, , micro, ...","[boat, deuc, usb, 300, 2, 1, typec, , micro, u..."
4,portronics konnect l 12m fast charging 3a 8 pi...,4.79,charge sync function this cable comes with ch...,"[charge, , sync, function, this, cable, comes,...","[charge, , sync, function, cable, comes, charg...","[charg, , sync, function, cabl, come, charg, ,...","[portronics, konnect, l, 12m, fast, charging, ...","[portronics, konnect, l, 12m, fast, charging, ...","[portron, konnect, l, 12m, fast, charg, 3a, 8,..."
...,...,...,...,...,...,...,...,...,...
1460,noir aqua 5pcs pp spun filter 1 spanner for...,11.03,supreme quality 90 gram 3 layer thik pp spun f...,"[supreme, quality, 90, gram, 3, layer, thik, p...","[supreme, quality, 90, gram, 3, layer, thik, p...","[suprem, qualiti, 90, gram, 3, layer, thik, pp...","[noir, aqua, , 5pcs, pp, spun, filter, , 1, sp...","[noir, aqua, , 5pcs, pp, spun, filter, , 1, sp...","[noir, aqua, , 5pc, pp, spun, filter, , 1, spa..."
1461,prestige delight prwo electric rice cooker 1 l...,36.54,230 volts 400 watts 1 year,"[230, volts, 400, watts, 1, year]","[230, volts, 400, watts, 1, year]","[230, volt, 400, watt, 1, year]","[prestige, delight, prwo, electric, rice, cook...","[prestige, delight, prwo, electric, rice, cook...","[prestig, delight, prwo, electr, rice, cooker,..."
1462,bajaj majesty rx10 2000 watts heat convector r...,36.96,international design and stylingtwo heat setti...,"[international, design, and, stylingtwo, heat,...","[international, design, stylingtwo, heat, sett...","[intern, design, stylingtwo, heat, set, 1000, ...","[bajaj, majesty, rx10, 2000, watts, heat, conv...","[bajaj, majesty, rx10, 2000, watts, heat, conv...","[bajaj, majesti, rx10, 2000, watt, heat, conve..."
1463,havells ventil air dsp 230mm exhaust fan pista...,22.68,fan sweep area 230 mm noise level 40 45 db f...,"[fan, sweep, area, 230, mm, , noise, level, 40...","[fan, sweep, area, 230, mm, , noise, level, 40...","[fan, sweep, area, 230, mm, , nois, level, 40,...","[havells, ventil, air, dsp, 230mm, exhaust, fa...","[havells, ventil, air, dsp, 230mm, exhaust, fa...","[havel, ventil, air, dsp, 230mm, exhaust, fan,..."


## ✅ Step 6: Final Preview


In [22]:
products_cleaned.head()


,product_name,actual_price,about_product
0,wayona nylon braided usb to lightning fast cha...,13.19,high compatibility compatible with iphone 12 ...
1,ambrane unbreakable 60w 3a fast charging 15m ...,4.19,compatible with all type c enabled devices be ...
2,sounce fast phone charging cable data sync us...,22.79,【 fast charger data sync】with builtin safety p...
3,boat deuce usb 300 2 in 1 typec micro usb str...,8.39,the boat deuce usb 300 2 in 1 cable is compati...
4,portronics konnect l 12m fast charging 3a 8 pi...,4.79,charge sync function this cable comes with ch...


In [23]:
products_cleaned.shape[0]

1465